# Intersectional Demographic Analysis
Breaking down that data by key demographics like age, gender, education, and geography (settlement type).
To start with analysis of demographic dimensions, I need to extract specific data points from `stg_eurostat_filtered` table. This extraction will generate several tables (one per each research question):
1. `stg_dig_skills_demog`: describes the relationships between different digital skills metrics and basic demographic dimensions (age, gender, education, level of urbanization); data available for 2025;
2. `stg_gov_demog_2025`/`stg_gov_demog_old`: relationships between e-Governence and eID and demographic dimensions;
3. `dtg_udage_demog`:

## Analytical Scope & Demographic Framework
To evaluate inequality and nuance across the digital landscape, this project captures multi-dimensional cross-sectional matrices across 15 highly granular socio-demographic slices defined by Eurostat. By structuring indicators across these specific intersections, the pipeline transforms flat records into an analytical framework capable of profiling systemic differences in digital engagement:
- **Gender Segregation** (`F_Y16_74`, `M_Y16_74`): Isolates broad digital participation gaps between female and male populations across the working-age band.
- **Digital Literacy & Skills Depth** (`I0_2`, `I3_4`, `I5_8`): Segregates populations based on documented digital proficiency scales—tracking individuals with low, medium, or high overall capability levels.
- **Urbanization & Infrastructure** (`IND_DEG1`, `IND_DEG2`, `IND_DEG3`): Groups individuals by geographic density (Cities, Towns/Suburbs, and Rural areas) to expose infrastructural, regional, and access-point divides.
- **Generational Cohorts** (`Y16_19` through `Y65_74`): Tracks continuous age bands to monitor generational trends, highlighting how advanced use cases (like AI adoption) scale among younger cohorts compared to structural exclusion patterns facing aging populations.




## Statistical Approach: Paired T-Test
Since in my dataframe I don't have columns 'gender' or 'age' or 'education' or 'urbanization level', I can't run normal correlation on my data. Instead, in order to check the effect of a certain demographic dimension (e.g., gender), I will treat the male percentages and female percentages for same indicator across 27 EU countries as **paired samples**. I will be testing distributioon of that indicator (e.g., AI usage) and if it changes significantly switching from the male group to female group.

The idea is to use `scipy.stats` for this:
```python
import scipy.stats as stats
# check if the difference between male and female AI usage across Europe is statistically significant
t_stat, p_value = stats.ttest_rel(df['i_iuai_m_y16_74'], df['i_iuai_f_y16_74'])
print(f"P-value: {p_value}")
```
If p-value < 0.05, there is a statistical proof that gender is significantly related to AI usage acrtoss the EU.

Another approach would be to melt tables 'on the fly' in `pandas` to long format (e.g., for gender it will be 27 countries * 2 genders, so 54 rows), convert gender string to a binary indicator (0 for Female, 1 for Male), and then run the classic correlation:
```python
# melt table:
long_df = df.melt(
    id_vars=['country_code'], 
    value_vars=['i_iuai_m_y16_74', 'i_iuai_f_y16_74'],
    var_name='gender', 
    value_name='ai_usage'
)

# convert gender string to 0s and 1s:
long_df['is_male'] = long_df['gender'].map({'i_iuai_m_y16_74': 1, 'i_iuai_f_y16_74': 0})

# correlation:
gender_correlation = long_df['is_male'].corr(long_df['ai_usage'])
print(f"Correlation between being male and AI usage: {gender_correlation}")
```

## 1. Digital Skills Demographic Model
Model: `stg_dig_skills_demog`

**Description**:   
This model tracks comprehensive digital literacy and competence profiles across various socio-demographic groups (gender, age, education level, and urbanization) in the EU.

**Why this matters:**  
Because Eurostat achieved perfect data coverage across all 27 countries and demographic slices for this specific module in 2025, this table contains zero historical time lags. It provides an airtight, high-density matrix optimized for modern clustering algorithms, correlation analysis, and population profiling in the Streamlit interface.

**EU DigComp (The Digital Competence Framework for Citizens)**

The EU uses this framework to establish policy targets (like the Digital Decade goal of having 80% of EU citizens possess at least basic digital skills by 2030) and to design the exact Eurostat surveys you are analyzing.

The framework splits digital competence into **five core components** (competence areas). Here is the breakdown of what they are and what they actually measure:
1. **Information and Data Literacy** measures an individual's ability to navigate the massive influx of online content. It focuses on articulate data gathering and critical thinking.
2. **Communication and Collaboration** assesses how people interact, share, and participate in society using digital platforms.
3. **Digital Content Creation** looks at the shift from being a passive consumer of the internet to an active creator.
4. **Safety** focuses on risk mitigation, security, and well-being in digital environments.
5. **Problem Solving** measures an individual’s agility and independence when things go wrong or when new tech emerges.

For the analytical purposes, this model keeps only overall digital skills levels (without going into details on each component).


**Core Metrics**:
- `I_DSK2_AB`: Individuals with above basic overall digital skills (all five component indicators are at above basic level).
- `I_DSK2_B`: Individuals with basic overall digital skills (all five component indicators are at basic or above basic level, without being all above basic).
- `I_DSK2_LM`: Individuals with limited overall digital skills (two out of five component indicators are at basic or above basic level).
- `I_DSK2_LW`: Individuals with low overall digital skills (four out of five component indicators are at basic or above basic level).
- `I_DSK2_N`: Individuals with narrow overall digital skills (three out of five component indicators are at basic or above basic level).
- `I_DSK2_X`: Individuals with no overall digital skills.
- `I_DSK2_NA`: Digital skills could not be assessed because the individual has not used the internet in the last 3 months.

## 2. Digital Government and eID Metrics
Models: `stg_gov_demog_2025`/`stg_gov_demog_old`

**Description**:  
These models track socio-demographic engagement with digital public administration and electronic identification (eID) options across the EU.
To prevent "temporal cross-contamination" caused by Eurostat's asynchronous survey rotation cycles, the data is split into two distinct wide tables:
- `stg_gov_demog_2025`: Captures a pure, synchronized snapshot of modern metrics collected during the 2025 survey release (e.g., active eID usage).
- `stg_gov_demog_old`: Isolates legacy administrative metrics from older collection cycles (2021–2024), preventing outdated data from biasing modern comparisons.

**Why this matters:**. 
This architectural split allows the downstream Streamlit app to cleanly switch between true current cross-sections and historical legacy trends without mixing pre- and post-pandemic realities in a single analytical row.

**Core Indicators**:
- `I_IGOV12FM`: Internet use: downloading official forms (last 12 months)

- `I_IGOV12IF`: Internet use: obtaining information from public authorities web sites (last 12 months).

- `I_IIGOVX`: Internet use: no issue when using a website or app of public authorities (last 12 months).

- `I_IUID1X`: Individuals have not used any electronic identification procedure for accessing online services (3 months)

- `I_IGOVTAX2`: Internet use: submitting my tax declaration (in the last 12 months) (as of 2024) 

## 3. Internet Usage Demographic Model 
Model: `stg_internet_usage_demog`

**Description**: 
This staging model establishes a comprehensive baseline of general internet frequency, access, and specific utility behaviors across 15 distinct socio-demographic slices in the EU.

**Scope**:  
Captures primary access metrics (such as daily usage patterns) alongside advanced digital interactions—including AI consumption (`I_IUAI`), civic or political participation via Internet (`I_IUCPP`), playing or downloading games (`I_IUPDG`), and difficulties encountered when using the internet (`I_IUPS`).

**Core Indicators Tracked** 
- `I_IDAY`: Daily internet users (frequency of internet access: daily).
- `I_IUAI`: Use of generative AI tools: in the last 3 months.
- `I_IUCPP`: Civic or political participation via Internet.
- `I_IUPDG`: Playing or downloading games.
- `I_IUPS`: Difficulties encountered when using the internet.
- `I_IUX`: Internet use: never.
- `I_UDI`: Individuals have seen untrue or doubtful information or content on the internet news sites or social media (3 months)
- `I_IUCHAT1`: Internet use: instant messaging, i.e. exchanging messages, for example, via Skype, Messenger, WhatsApp, Viber
- `I_IUPOL2`: Internet use: expressing opinions on civic or political issues on websites or in social media (e.g. Facebook, Twitter, Instagram, YouTube) 